# 14.4 - Execution

Status: VERIFIED

## What Are We Solving?
Execution is carrying out each planned step by invoking tools, collecting results, and updating state. A plan without execution is just a list.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Execution Engine

In [2]:
class ExecutionEngine:
    def __init__(self):
        self.tools = {}
        self.results = []
    
    def register_tool(self, name: str, func):
        self.tools[name] = func
    
    def execute_step(self, step: dict) -> dict:
        tool_name = step.get("tool", "unknown")
        args = step.get("args", {})
        
        if tool_name not in self.tools:
            return {"status": "error", "message": f"Unknown tool: {tool_name}"}
        
        try:
            result = self.tools[tool_name](**args)
            entry = {"step": step, "status": "success", "result": str(result)[:200]}
        except Exception as e:
            entry = {"step": step, "status": "error", "message": str(e)}
        
        self.results.append(entry)
        return entry

# Register tools
def read_data(source: str) -> str:
    return f"Data from {source}: [1, 2, 3, 4, 5]"

def compute_stats(data: str) -> str:
    nums = [int(x) for x in data.split(":")[-1].strip("[]").split(",")]
    return json.dumps({"mean": sum(nums)/len(nums), "min": min(nums), "max": max(nums)})

def save_result(content: str) -> str:
    return f"Saved: {content[:50]}"

engine = ExecutionEngine()
engine.register_tool("read_data", read_data)
engine.register_tool("compute_stats", compute_stats)
engine.register_tool("save_result", save_result)

# Execute a pipeline
steps = [
    {"tool": "read_data", "args": {"source": "sales.csv"}},
    {"tool": "compute_stats", "args": {"data": "Data from sales.csv: [1, 2, 3, 4, 5]"}},
    {"tool": "save_result", "args": {"content": '{"mean": 3.0}'}},
]

print("Execution trace:")
for step in steps:
    result = engine.execute_step(step)
    status = result["status"]
    detail = result.get("result", result.get("message"))
    print(f"  {step['tool']}: {status} -> {detail[:80]}")

Execution trace:


  read_data: success -> Data from sales.csv: [1, 2, 3, 4, 5]
  compute_stats: error -> invalid literal for int() with base 10: ' [1'
  save_result: success -> Saved: {"mean": 3.0}


## Error Handling

In [3]:
# Test error handling
bad_steps = [
    {"tool": "unknown_tool", "args": {}},
    {"tool": "read_data", "args": {"source": "test.csv"}},
]

for step in bad_steps:
    result = engine.execute_step(step)
    print(f"  {step['tool']}: {result['status']} -> {result.get('result', result.get('message'))[:80]}")

print(f"\nTotal results logged: {len(engine.results)}")

  unknown_tool: error -> Unknown tool: unknown_tool
  read_data: success -> Data from test.csv: [1, 2, 3, 4, 5]

Total results logged: 4


In [4]:
# Verification
assert len(engine.results) > 0, "Must have results"
assert any(r["status"] == "error" for r in engine.results), "Must handle errors"
print("VERIFICATION PASSED: Phase 14.4 complete")

VERIFICATION PASSED: Phase 14.4 complete
